# Setup
For this project, we adopt the following convention for units:
- AU (astronomical units) for distance
- days for time
- solar mass for mass

In [32]:
import numpy as np
import matplotlib.pyplot as plt

# physical constants in units of AU, days and solar-mass.
GM_SUN   = 2.9591220828559115e-4   # G * M_sun in AU^3 day^-2   
C_LIGHT  = 173.14463348            # AU day^-1

# unit conversions
ARCSEC   = 206264.806              # arcsec per radian
DAY_YEAR = 365.25                  

T_MERCURY = 87.969                 # mercury orbital period (in days)
ORBITS_PER_CENTURY = 100.0 * DAY_YEAR / T_MERCURY  # ~415.2 mercury orbits per century

print(f"GM_sun = {GM_SUN:.6e}  AU^3/day^2")
print(f"c      = {C_LIGHT:.5f}  AU/day")
print(f"Mercury: {ORBITS_PER_CENTURY:.2f} orbits per century")


GM_sun = 2.959122e-04  AU^3/day^2
c      = 173.14463  AU/day
Mercury: 415.20 orbits per century


# NASA HORIZONS initial conditions (sun + 8 planets)

Source: Standish (2001) "Keplerian Elements for Approximate Positions of the Major Planets"  https://ssd.jpl.nasa.gov/planets/approx_pos.html

We obtain our data from the above source. However, the data is provided in terms of orbital elements:
- $a$ / `a`: semi-major axis (AU)
- $e$ / `e`: eccentricity
- $i$ / `i`: inclination (deg)
- $\Omega$ / `Omega`: longitude of ascending node (deg)
- $\varpi$ / `varpi`: longitude of perihelion (deg), ie direction of perihelion in reference frame
- $L$ / `L`: mean longitude (deg)

The first objective is to convert these orbital parameters to heliocentric (Sun at origin) Cartesian position and velocity vectors

### Convert orbital parameters to heliocentric Cartesian vectors

In [33]:
def get_orbital_angles(i_deg, Omega_deg, varpi_deg, L_deg):
    # data for each planet is provided in degrees. this function converts them to radians
    # this function also provides omega = varpi - Omega, the argument of perihelion,
    # as well as M, the mean anomaly of the planet.
    deg_to_rad = np.pi / 180.0
    i = i_deg * deg_to_rad
    Omega = Omega_deg * deg_to_rad
    omega = (varpi_deg - Omega_deg) * deg_to_rad
    M = ((L_deg - varpi_deg) % 360.0) * deg_to_rad
    return i, Omega, omega, M

def solve_kepler(M, e):
    # kepler's equation is M = E - e sin(E), where E is the eccentric anomaly.
    # we want E, but this equation cannot be rearranged analytically
    # hence, we use NR to find the root of E - e sin(E) - M = 0
    # our initial guess is E = M as e < 1

    E = M
    for _ in range(100):
        change = (M - E + e*np.sin(E)) / (1 - e*np.cos(E)) # this term is just -f(E)/f'(E)
        E += change
        if abs(change) < 1e-12:
            break
    return E

def orbital_plane_state(a, e, f, gm):
    # this function first finds position and velocity of the orbit in a 2D plane
    # f is the angle of the planet on its ellipse, measured from perihelion (closest point to sun)

    p = a * (1 - e**2) # computes the semi-latus rectum, an orbital parameter
    r = p / (1 + e*np.cos(f)) # distance from sun
    speed_factor = np.sqrt(gm / p)

    # converts polar coordinates to cartesian
    x_orb = r * np.cos(f)
    y_orb = r * np.sin(f)

    vx_orb = -speed_factor * np.sin(f)
    vy_orb = speed_factor * (e + np.cos(f))

    return x_orb, y_orb, vx_orb, vy_orb

def rotate_to_xyz(x_orb, y_orb, vx_orb, vy_orb, i, Omega, omega):
    # this function takes our orbits defined in 2D planes and rotates them into 3D space

    # precompute trig evaluations 
    cO = np.cos(Omega)
    sO = np.sin(Omega)
    co = np.cos(omega)
    so = np.sin(omega)
    ci = np.cos(i)
    si = np.sin(i)

    # these are the components of the rotation matrix to be applied to the 2D position and velocity
    # more specifically, P and Q are the directions of the x and y axes of the orbital plane in 3D
    Px =  cO*co - sO*so*ci
    Py =  sO*co + cO*so*ci
    Pz =  so*si

    Qx = -cO*so - sO*co*ci
    Qy = -sO*so + cO*co*ci
    Qz =  co*si

    P = np.array([Px, Py, Pz])
    Q = np.array([Qx, Qy, Qz])

    # rotates our 2D position and velocity into 3D space
    pos = x_orb * P + y_orb * Q
    vel = vx_orb * P + vy_orb * Q

    return pos, vel

def elements_to_cartesian(a, e, i_deg, Omega_deg, varpi_deg, L_deg, gm=GM_SUN):
    # this function takes one planet's orbital elements and returns its 3D heliocentric position and velocity
    # it is a wrapper function that uses all above defined functions

    # convert degrees to radians
    i, Omega, omega, M = get_orbital_angles(i_deg, Omega_deg, varpi_deg, L_deg)

    # solve for E
    E = solve_kepler(M, e)

    # compute f, the true anomaly (angle of the planet measured from its perihelion)
    f = 2*np.arctan2(np.sqrt(1+e)*np.sin(E/2),
                     np.sqrt(1-e)*np.cos(E/2))

    # compute 2D position and velocity
    x_orb, y_orb, vx_orb, vy_orb = orbital_plane_state(a, e, f, gm)

    # compute position and velocity vectors in 3D
    pos, vel = rotate_to_xyz(x_orb, y_orb, vx_orb, vy_orb, i, Omega, omega)
    return pos, vel

### Data

In [34]:
# this table is each planet's orbital parameters at January 1, 2000, which we take as our initial time.
# the format of each entry is (planet name, a (AU), e, i (deg), Omega (deg), varpi (deg), L (deg))
ELEMENTS_J2000 = [
    ("Mercury", 0.38709927, 0.20563593,  7.00497,  48.33167,  77.45645, 252.25084),
    ("Venus",   0.72333566, 0.00677672,  3.39467,  76.67984, 131.60246, 181.97973),
    ("Earth",   1.00000261, 0.01671123,  0.00001,   0.0,     102.93768, 100.46457),
    ("Mars",    1.52371034, 0.09339410,  1.84969,  49.55953, 336.04084,  -4.55343),
    ("Jupiter", 5.20288700, 0.04838624,  1.30439, 100.47390,  14.72847,  34.39644),
    ("Saturn",  9.53667594, 0.05386179,  2.48599, 113.66242,  92.59887,  49.95424),
    ("Uranus",  19.18916464,0.04725744,  0.77263,  74.01693, 170.95427, 313.23218),
    ("Neptune", 30.06992276,0.00859048,  1.77004, 131.78422,  44.96476, -55.12002),
]

# specifies the colours of each planet for plotting
PLANET_COLORS = {
    "Mercury": '#b5b5b5', "Venus":   '#e8c890', "Earth":   '#4fa3e0',
    "Mars":    '#c1440e', "Jupiter": '#c88b3a', "Saturn":  '#e4d191',
    "Uranus":  '#7de8e8', "Neptune": '#3f54ba',
}

# our simulation uses mass ratio = GM_planet / GM_sun 
MASS_RATIOS = {    
    "Mercury": 1.6601e-7, "Venus":   2.4478e-6, "Earth":   3.0034e-6,
    "Mars":    3.2272e-7, "Jupiter": 9.5479e-4, "Saturn":  2.8589e-4,
    "Uranus":  4.3662e-5, "Neptune": 5.1514e-5,
}

# extracts a list of all planet names for indexing and plotting
PNAMES = [row[0] for row in ELEMENTS_J2000]


### Setup
Here we set up our initial state of the solar system

In [35]:
def planet_state_from_row(row):
    # takes a planet's data and converts it to heliocentric cartesian coordinates
    # as well as gravitational parameter GM

    a = row[1]
    e = row[2]
    i_deg = row[3]
    Omega_deg = row[4]
    varpi_deg = row[5]
    L_deg = row[6]

    pos, vel = elements_to_cartesian(a, e, i_deg, Omega_deg, varpi_deg, L_deg)
    gm = GM_SUN * MASS_RATIOS[row[0]]

    return pos, vel, gm

def get_sun_velocity(planet_velocities, planet_gm):
    # sets the sun's velocity so that the total momentum of the system is zero
    # sum(m_i v_i) = 0 -> v_sun = -sum(m_planet v_planet) / M_sun

    total_planet_momentum = np.sum(planet_gm[:, None] * planet_velocities, axis=0)
    sun_velocity = -total_planet_momentum / GM_SUN
    return sun_velocity

def pack_state():
    # this function builds the initial state vectors of the solar system
    # returns three arrays: posistions, velocities and GM values
    # with sun at index 0, then mercury to neptune

    planet_positions = []
    planet_velocities = []
    planet_gm = []

    for row in ELEMENTS_J2000:
        pos, vel, gm = planet_state_from_row(row)
        planet_positions.append(pos)
        planet_velocities.append(vel)
        planet_gm.append(gm)

    planet_positions = np.array(planet_positions)
    planet_velocities = np.array(planet_velocities)
    planet_gm = np.array(planet_gm)

    sun_position = np.array([0.0, 0.0, 0.0])
    sun_velocity = get_sun_velocity(planet_velocities, planet_gm)

    all_positions = np.vstack((sun_position, planet_positions))
    all_velocities = np.vstack((sun_velocity, planet_velocities))
    all_gm = np.concatenate(([GM_SUN], planet_gm))

    return all_positions, all_velocities, all_gm

### Checks

In [36]:
def orbital_elements(r_vec, v_vec, gm=GM_SUN):
    # converts r_vec and v_vec to a and e
    # used to verify if the generated vectors reproduce the intended orbit

    r   = np.linalg.norm(r_vec)
    v2  = v_vec @ v_vec

    eps = 0.5*v2 - gm/r
    a   = -gm/(2*eps)

    h   = np.cross(r_vec, v_vec) # angular momentum vector
    e   = np.sqrt(max(0.0, 1 + 2*eps*(h@h)/gm**2))
    return a, e

def system_energy(pos, vel, gm):
    # computes the total energy of the system
    # used to check how well total energy is conserved

    speeds_sq = np.sum(vel**2, axis=1)   # v_i^2 for each body
    KE = 0.5 * np.sum(gm * speeds_sq)

    PE = 0.0
    n = len(pos)

    for i in range(n):
        for j in range(i+1, n):
            r_ij = np.linalg.norm(pos[j] - pos[i])
            PE += -gm[i] * gm[j] / r_ij

    return KE + PE

def check_initial_conditions(pos, vel):
    # checks whether the generated vectors reproduce the intended eccentricities

    print(f"{'Planet':<10} {'|r| AU':>8} {'a_elem':>8} {'e_elem':>8} {'e_calc':>10} {'err%':>8}")
    print("-" * 60)

    for k, row in enumerate(ELEMENTS_J2000):
        name = row[0]
        a_true = row[1]
        e_true = row[2]

        idx = k + 1   # Sun is index 0
        r_hel = pos[idx] - pos[0]
        v_hel = vel[idx] - vel[0]

        _, e_calc = orbital_elements(r_hel, v_hel) # only comparing e, so we ignore the computed a
        err_percent = abs(e_calc - e_true) / max(e_true, 1e-9) * 100

        distance = np.linalg.norm(r_hel)

        print(f"{name:<10} {distance:8.4f} {a_true:8.5f} {e_true:8.6f} {e_calc:10.8f} {err_percent:8.4f}%")

    print(f"\nSun barycentric speed: {np.linalg.norm(vel[0]):.4e} AU/day")

In [37]:
pos0, vel0, gm0 = pack_state()
check_initial_conditions(pos0, vel0)

Planet       |r| AU   a_elem   e_elem     e_calc     err%
------------------------------------------------------------
Mercury      0.4665  0.38710 0.205636 0.20613030   0.2404%
Venus        0.7202  0.72334 0.006777 0.00653535   3.5617%
Earth        0.9833  1.00000 0.016711 0.01715708   2.6679%
Mars         1.3912  1.52371 0.093394 0.09421393   0.8778%
Jupiter      4.9673  5.20289 0.048386 0.05065990   4.6990%
Saturn       9.1723  9.53668 0.053862 0.05597209   3.9180%
Uranus      19.9216 19.18916 0.047257 0.04529386   4.1551%
Neptune     30.1173 30.06992 0.008590 0.01149836  33.8500%

Sun barycentric speed: 9.1568e-06 AU/day


# Calculating Acceleration
## Using Newton's Laws
The function below takes in the $i$-th body's position and its gravitational parameter and finds its acceleration:
$$
\vec a_i = \sum_{j\neq i}\frac{GM_j}{r^3_{ij}} (\vec r_j - \vec r_i),
$$
where $r_{ij} = |\vec r_j - \vec r_i|$

In [38]:
def newtonian_acc(pos, gm):
    # computes newtonian gravitational acceleration for all bodies
    # pos[i] = position of body i, stored as a triple (r_x, r_y, r_z)
    # gm[i]  = GM of body i

    n = len(pos)
    acc = np.zeros((n, 3)) # n x 3 matrix

    for i in range(n):
        for j in range(n):
            if i != j:
                r_ij = pos[j] - pos[i]              # 3D vector from i to j
                distance = np.linalg.norm(r_ij)

                acc[i] += gm[j] * r_ij / distance**3

    return acc

## EIH Correction
Here we compute the first post-Newtonian Einstein-Infeld-Hoffman (EIH) correction to the Newtonian gravitational acceleration of the $i$-th body:
$$
\vec a_i = \vec a_i^{\text{Newton}} + \vec a_i^{\text{EIH}}
$$
This allows us to model effects such as Mercury's perihelion precession

The correction is given by
$$
\vec a_i^{\text{EIH}} = \frac1 {c^2}\sum_{j \neq i} \frac{GM_j}{r^3_{ij}}\left[A_{ij}\vec r_{ij} + B_{ij} (\vec v_i - \vec v_j)\right].
$$

To describe the components of this equation, we define
$$
\Phi_i = \sum_{k \neq i} \frac{GM_k}{r_{ik}}.
$$

The components are hence:
$$
\begin{aligned}
\vec r_{ij} &= \vec r_j - \vec r_i \\
A_{ij}      &= -v^2_i + 4\vec v_i\cdot\vec v_j-2v_j^2+\frac 32\left(\frac{\vec r_{ij}\cdot\vec v_j}{r_{ij}}\right)^2 + 4\Phi_i - \Phi_j \\
B_{ij}      &= 4(\vec r_{ij}\cdot\vec v_i )-3(\vec r_{ij}\cdot \vec v_j)
\end{aligned}


In [39]:
def eih_correction(pos, vel, gm):

    n = len(pos)
    c_sq = C_LIGHT**2
    mask = ~np.eye(n, dtype=bool)

    # pairwise displacement vectors and distances
    dr = pos[np.newaxis, :, :] - pos[:, np.newaxis, :]
    r = np.linalg.norm(dr, axis=2)
    r_safe = np.where(mask, r, 1.0)

    # phi_i = sum_{k != i} GM_k / r_ik
    phi = np.sum(np.where(mask, gm[np.newaxis, :] / r_safe, 0.0), axis=1)

    # velocity-related quantities
    v_sq = np.sum(vel**2, axis=1)
    vi_dot_vj = vel.dot(vel.T)
    rij_dot_vj = np.sum(dr * vel[np.newaxis, :, :], axis=2)
    rij_dot_vi = np.sum(dr * vel[:, np.newaxis, :], axis=2)

    # scalar coefficients in the EIH formula
    A = (-v_sq[:, np.newaxis]
         + 4 * vi_dot_vj
         - 2 * v_sq[np.newaxis, :]
         + 1.5 * (rij_dot_vj / r_safe)**2
         + 4 * phi[:, np.newaxis]
         - phi[np.newaxis, :])

    B = 4 * rij_dot_vi - 3 * rij_dot_vj

    # common newtonian-style weight and pairwise velocity difference
    weight = np.where(mask, gm[np.newaxis, :] / r_safe**3, 0.0)
    dv = vel[:, np.newaxis, :] - vel[np.newaxis, :, :]

    # sum pairwise correction terms over j
    # the overall sign gives the attractive relativistic correction
    correction = -(
        (weight * A)[:, :, np.newaxis] * dr
        + (weight * B)[:, :, np.newaxis] * dv
    ).sum(axis=1) / c_sq

    return correction

# Integrators

In [40]:
def compute_acc(pos, vel, gm, use_eih=False):
    # computes total acceleration: newtonian + optional EIH correction
    acc = newtonian_acc(pos, gm)

    if use_eih:
        acc += eih_correction(pos, vel, gm)

    return acc


def euler_step(pos, vel, acc, gm, dt, use_eih=False):
    # advances the system by one euler step

    pos_new = pos + dt * vel
    vel_new = vel + dt * acc
    acc_new = compute_acc(pos_new, vel_new, gm, use_eih)

    return pos_new, vel_new, acc_new


def leapfrog_step(pos, vel, acc, gm, dt, use_eih=False):
    # advances the system by one velocity leapfrog step

    pos_new = pos + dt * vel + 0.5 * dt**2 * acc

    # predictor velocity is used because the EIH correction depends on velocity
    vel_pred = vel + dt * acc
    acc_new = compute_acc(pos_new, vel_pred, gm, use_eih)

    vel_new = vel + 0.5 * dt * (acc + acc_new)

    # recompute acceleration using the corrected velocity
    acc_new = compute_acc(pos_new, vel_new, gm, use_eih)

    return pos_new, vel_new, acc_new


def integrate(pos0, vel0, gm, dt, n_steps, save_every=1,
              method='leapfrog', use_eih=False, progress_every=30000):
    # evolves the N-body system forward in time
    # returns saved positions, velocities, and total energy

    pos = pos0.copy()
    vel = vel0.copy()

    n_bodies = len(pos)
    n_save = n_steps // save_every + 1

    tr = np.zeros((n_save, n_bodies, 3))  # array of positions
    tv = np.zeros((n_save, n_bodies, 3))  # array of velocities
    te = np.zeros(n_save)                 # array of energies

    tr[0] = pos
    tv[0] = vel
    te[0] = system_energy(pos, vel, gm)

    acc = compute_acc(pos, vel, gm, use_eih)
    save_index = 0

    for step in range(1, n_steps + 1):
        if method == 'euler':
            pos, vel, acc = euler_step(pos, vel, acc, gm, dt, use_eih)

        elif method == 'leapfrog':
            pos, vel, acc = leapfrog_step(pos, vel, acc, gm, dt, use_eih)

        else:
            raise ValueError("method must be 'euler' or 'leapfrog'")

        # it is not necessary to store values for every single time step
        if step % save_every == 0:
            save_index += 1
            tr[save_index] = pos
            tv[save_index] = vel
            te[save_index] = system_energy(pos, vel, gm)

        # displays how far through the simulation we are
        if progress_every and step % progress_every == 0:
            percent = 100 * step / n_steps
            print(f"  {percent:5.1f}%  ({step:,}/{n_steps:,} steps)", end="\r")

    print()
    return tr, tv, te